# Análises complementares

1. **Série anual da saúde** (regra R2).
2. **Razão de chances da Camada 2**, com e sem controle de comprimento e de legislatura.
3. **Termos que mais contribuem** para cada categoria setorizada, prevalência sem termos genéricos e 50 trechos de religião e moral para leitura qualitativa.
4. **Frequência de coalizão, impeachment e stf** por legislatura.

**Dados de entrada:** corpus pré-processado em `legislatura/data/artigo1_preprocessado`, gerado por `scripts/01_preprocess_corpus.py` a partir dos dados da API da Câmara dos Deputados (ver `data/README_DATA.md`). O notebook procura essa pasta no Google Drive montado.

**Execução:** Google Colab → *Ambiente de execução → Executar tudo*. Tempo estimado: 15 a 20 minutos.


In [ ]:
# 1) Conectar o Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Localizar a pasta pré-processada da dissertação
from pathlib import Path
RAIZ = Path('/content/drive/MyDrive')
candidatos = [RAIZ / 'legislatura' / 'data' / 'artigo1_preprocessado']
candidatos += list(RAIZ.glob('*/legislatura/data/artigo1_preprocessado'))
candidatos += list(RAIZ.glob('*/data/artigo1_preprocessado'))
PRE = next((c for c in candidatos if (c / 'corpus_artigo1_metadata.csv').exists()), None)
if PRE is None:
    raise SystemExit('Pasta artigo1_preprocessado não encontrada no Drive montado (ver Dados de entrada no topo).')
print('Pasta encontrada:', PRE)

SAIDA = RAIZ / 'MESTRADO' / 'resultados_pendencias_finais'
SAIDA.mkdir(parents=True, exist_ok=True)
PERCENTIL_R3 = 0.50   # percentil do limiar fixo da R3 (0.50 = mediana)

In [ ]:
# 3) Ler os discursos pré-processados (mesma leitura do script 14)
import pandas as pd
from collections import Counter
LEGS = [52, 53, 54, 55, 56, 57]
meta = pd.read_csv(PRE / 'corpus_artigo1_metadata.csv', usecols=['id_legislatura'])
textos, legs = [], []
for leg in LEGS:
    linhas = (PRE / 'by_legislature' / f'leg{leg}.txt').read_text(encoding='utf-8').splitlines()
    n_meta = int((meta['id_legislatura'] == leg).sum())
    if n_meta != len(linhas):
        raise ValueError(f'Leg {leg}: metadados={n_meta} texto={len(linhas)}')
    textos += linhas; legs += [leg] * len(linhas)
    print(f'Legislatura {leg}: {len(linhas):,} discursos')
df = pd.DataFrame({'_leg': legs, '_tokens': [t.split() for t in textos]})
del textos, meta
print(f'TOTAL: {len(df):,} discursos  (dissertação: 439.627)')

In [ ]:
# 4) Dicionários
SETORIZADO = {
 "agricultura_rural": {"agricultura","agricola","agropecuaria","agronegocio","produtor","produtores","rural","campo","pecuaria","safra","assentamento"},
 "empresas_industria": {"empresa","empresas","industria","industrial","comercio","comerciante","empreendedor","empreendedores","microempresa","empresarios"},
 "trabalhadores_sindicatos": {"trabalhador","trabalhadores","sindicato","sindicatos","servidor","servidores","categoria","categorias","funcionario","funcionarios"},
 "servidores_profissoes": {"professor","professores","medico","medicos","enfermeiro","enfermeiros","policial","policiais","militar","militares"},
 "mulheres_familia_infancia": {"mulher","mulheres","familia","familias","crianca","criancas","adolescente","adolescentes","juventude","idosos","idoso"},
 "raca_indigenas_minorias": {"negros","negro","racismo","indigena","indigenas","quilombolas","quilombola","lgbt","deficiencia","pessoa","pessoas"},
 "religiao_moral": {"religiao","religioso","religiosos","igreja","igrejas","evangelicos","catolicos","familia","vida","aborto"},
 "territorio_regiao": {"municipio","municipios","prefeito","prefeitos","nordeste","norte","amazonia","semiarido","regional","regiao","comunidade","comunidades"},
 "forcas_seguranca": {"policia","policial","policiais","militar","militares","bombeiros","guarda","forcas","armadas"},
}
TERMOS_SET = set().union(*SETORIZADO.values())

NAC_A = {
 "educacao": {"educacao","escola","ensino","professor","professores","universidade","fundeb"},
 "saude": {"saude","sus","hospital","medico","medicos","vacina","pandemia","covid"},
 "economia": {"economia","emprego","trabalho","renda","salario","inflacao","juros","credito"},
 "seguranca": {"seguranca","violencia","crime","policia","armas","trafico","homicidio"},
 "infraestrutura": {"infraestrutura","transporte","rodovia","estrada","energia","saneamento","internet"},
 "meio_ambiente": {"ambiente","ambiental","amazonia","floresta","desmatamento","clima","agua"},
 "direitos_politica_social": {"direitos","cidadania","pobreza","fome","desigualdade","assistencia","previdencia"},
 "instituicoes": {"democracia","constituicao","congresso","camara","senado","governo","voto","judiciario"},
}
NAC_B = {k: v - TERMOS_SET for k, v in NAC_A.items()}

CAMADA1 = {
 "Educação": {"educacao","escola","escolas","ensino","professor","professores","aluno","alunos","universidade","universidades","creche","alfabetizacao","fundeb","enem","estudante","estudantes","pesquisa","ciencia","cientifico","tecnologia"},
 "Saúde": {"saude","sus","hospital","hospitais","medico","medicos","enfermeiro","enfermeiros","enfermeiras","vacina","vacinas","pandemia","covid","doenca","doencas","tratamento","medicamento","medicamentos","pacientes","atendimento"},
 "Economia, trabalho e renda": {"economia","economico","economica","emprego","empregos","trabalho","trabalhador","trabalhadores","renda","salario","salarios","inflacao","juros","credito","industria","empresas","empresa","mercado","crescimento","desenvolvimento","investimento","investimentos","tributaria","imposto","impostos"},
 "Segurança pública e violência": {"seguranca","violencia","crime","crimes","criminalidade","policia","policial","policiais","prisao","presidio","armas","arma","trafico","homicidio","homicidios","feminicidio","militar","militares"},
 "Meio ambiente e clima": {"meio","ambiente","ambiental","amazonia","floresta","florestas","desmatamento","clima","climatica","climatico","sustentavel","sustentabilidade","biodiversidade","indigena","indigenas","garimpo","queimadas","agua","saneamento"},
 "Infraestrutura, transporte e habitação": {"infraestrutura","transporte","transportes","rodovia","rodovias","estrada","estradas","ferrovia","ferrovias","porto","portos","aeroporto","aeroportos","energia","eletrica","habitacao","moradia","saneamento","internet","telecomunicacoes","obra","obras"},
 "Proteção social e desigualdade": {"pobreza","fome","desigualdade","desigualdades","assistencia","social","bolsa","familia","beneficio","beneficios","previdencia","aposentadoria","aposentados","vulnerabilidade","moradia","direitos","mulheres","criancas","juventude","idosos","deficiencia"},
 "Democracia e instituições": {"democracia","democratico","democratica","constituicao","constitucional","congresso","camara","senado","parlamento","governo","presidente","presidencia","republica","eleicao","eleicoes","voto","impeachment","stf","supremo","judiciario","justica"},
 "Direitos e cidadania": {"direito","direitos","cidadania","cidadao","cidadaos","igualdade","liberdade","justica","mulher","mulheres","negros","racismo","discriminacao","lgbt","indigena","indigenas","quilombolas","criancas","adolescentes"},
}
NAC_C = {k: v - TERMOS_SET for k, v in CAMADA1.items()}

TERMOS = {"A": set().union(*NAC_A.values()), "B": set().union(*NAC_B.values()), "C": set().union(*NAC_C.values())}
print("Termos retirados em B:", sorted(TERMOS["A"] - TERMOS["B"]))
print("Termos de agenda nacional -> A:", len(TERMOS["A"]), "| B:", len(TERMOS["B"]), "| C:", len(TERMOS["C"]))

In [ ]:
# 5) Preparar ano e tamanho de cada discurso
import numpy as np
pd.set_option("display.max_colwidth", 250)
pd.set_option("display.max_rows", 100)
meta_d = pd.read_csv(PRE / "corpus_artigo1_metadata.csv", usecols=["id_legislatura", "data"])
anos = []
for L in LEGS:
    anos += list(pd.to_datetime(meta_d.loc[meta_d["id_legislatura"] == L, "data"], errors="coerce").dt.year)
df["_ano"] = anos
df["_n"] = df["_tokens"].map(len)
print("Pronto:", len(df), "discursos")

In [ ]:
# 6) PENDÊNCIA 1 — Série anual da saúde (R2)
SAUDE = CAMADA1["Saúde"]
def r2_cat(toks, termos):
    cont = Counter(t for t in toks if t in termos)
    return len(cont) >= 2 or sum(cont.values()) >= 3
df["_saude"] = [r2_cat(t, SAUDE) for t in df["_tokens"]]
print("Conferência por legislatura (56ª deve dar perto de 18,3):")
print(df.groupby("_leg")["_saude"].mean().mul(100).round(1).to_string())
p1 = df.groupby("_ano").agg(discursos=("_saude", "size"),
                            saude_pct=("_saude", lambda x: round(100 * x.mean(), 1)),
                            tokens_medio=("_n", "mean")).round(1)
p1.to_csv(SAIDA / "p1_saude_por_ano.csv", encoding="utf-8-sig")
p1

In [ ]:
# 7) PENDÊNCIA 2 — Razão de chances da Camada 2
import warnings; warnings.filterwarnings("ignore")
import statsmodels.formula.api as smf
TN = TERMOS["A"]
nac = np.array([any(t in TN for t in toks) for toks in df["_tokens"]])
sett = np.array([any(t in TERMOS_SET for t in toks) for toks in df["_tokens"]])
print("Setorizada em todos os discursos:", round(100 * sett.mean(), 1))
print("Setorizada entre os COM agenda nacional:", round(100 * sett[nac].mean(), 1))
print("Setorizada entre os SEM agenda nacional:", round(100 * sett[~nac].mean(), 1))
d = pd.DataFrame({"s": sett.astype(int), "n": nac.astype(int), "lt": np.log(df["_n"].clip(lower=1)), "leg": df["_leg"]})
for nome, f in [("sem controle", "s ~ n"),
                ("com controle de comprimento", "s ~ n + lt"),
                ("com comprimento e legislatura", "s ~ n + lt + C(leg)")]:
    m = smf.logit(f, d).fit(disp=0)
    ic = np.exp(m.conf_int().loc["n"]).round(2).tolist()
    print(nome, "| razão de chances =", round(float(np.exp(m.params["n"])), 2), "| IC 95%:", ic)

In [ ]:
# 8) PENDÊNCIA 3 — Termos que mais contribuem e termos genéricos
GENERICOS = {"raca_indigenas_minorias": {"pessoa", "pessoas"},
             "religiao_moral": {"vida", "familia"},
             "mulheres_familia_infancia": {"familia", "familias"}}
linhas = []
for cat, termos in SETORIZADO.items():
    flag = np.array([any(t in termos for t in toks) for toks in df["_tokens"]])
    cont = Counter()
    for toks, f in zip(df["_tokens"], flag):
        if f:
            cont.update(set(toks) & termos)
    nflag = max(int(flag.sum()), 1)
    top = "; ".join(f"{t} ({100 * n / nflag:.0f}%)" for t, n in cont.most_common(5))
    linha = {"categoria": cat, "prevalencia_%": round(100 * flag.mean(), 1), "top5_termos (% dos sinalizados)": top}
    if cat in GENERICOS:
        sem = termos - GENERICOS[cat]
        flag2 = np.array([any(t in sem for t in toks) for toks in df["_tokens"]])
        linha["sem_genericos_%"] = round(100 * flag2.mean(), 1)
        linha["termos_retirados"] = ", ".join(sorted(GENERICOS[cat]))
    linhas.append(linha)
p3 = pd.DataFrame(linhas).set_index("categoria")
p3.to_csv(SAIDA / "p3_termos_por_categoria.csv", encoding="utf-8-sig")
p3

In [ ]:
# 9) PENDÊNCIA 3 (cont.) — 50 trechos de "religião e moral" para leitura manual
termos_rel = SETORIZADO["religiao_moral"]
idx = [i for i, toks in enumerate(df["_tokens"]) if any(t in termos_rel for t in toks)]
rng = np.random.default_rng(42)
amostra = rng.choice(idx, size=min(50, len(idx)), replace=False)
trechos = []
for i in amostra:
    toks = df["_tokens"].iat[i]
    k = next(j for j, t in enumerate(toks) if t in termos_rel)
    trechos.append({"legislatura": df["_leg"].iat[i], "termo": toks[k],
                    "trecho": " ".join(toks[max(0, k - 12): k + 13])})
p3b = pd.DataFrame(trechos)
p3b.to_csv(SAIDA / "p3_50_trechos_religiao.csv", encoding="utf-8-sig", index=False)
print(p3b["termo"].value_counts().to_string())
p3b

In [ ]:
# 10) PENDÊNCIA 4 — Frequência de coalizão, impeachment e stf
alvo = ["coalizao", "impeachment", "stf"]
linhas = []
for L in LEGS:
    sub = df.loc[df["_leg"] == L, "_tokens"]
    cont = Counter(t for toks in sub for t in toks if t in alvo)
    ntok = int(sub.map(len).sum())
    linha = {"legislatura": L}
    for t in alvo:
        linha[t + "_ocorrencias"] = cont[t]
        linha[t + "_por_milhao"] = round(1e6 * cont[t] / ntok, 1)
    linhas.append(linha)
p4 = pd.DataFrame(linhas).set_index("legislatura")
p4.to_csv(SAIDA / "p4_frequencia_termos.csv", encoding="utf-8-sig")
print("Arquivos salvos em:", SAIDA)
p4